<a href="https://colab.research.google.com/github/440g/painkiller/blob/main/src/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model training and validation
* Selection of multiple machine learning algorithms.

In [29]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [30]:
import pandas as pd

X_train = pd.read_csv('../datasets/X_train_re2.csv')
#파일 맨 앞에 의미없는 인덱스 값이 존재해 제거
# X_train = X_train.drop(columns=['Unnamed: 0'])
y_train = pd.read_csv('../datasets/y_train_re2.csv')
X_val = pd.read_csv('../datasets/X_train_val_re2.csv')
# X_val = X_val.drop(columns=['Unnamed: 0'])
y_val = pd.read_csv('../datasets/y_train_val_re2.csv')

In [31]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import KFold, GridSearchCV

In [32]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = "f1"

models = {}

###1. Decision Tree

In [33]:
model = DecisionTreeClassifier(random_state=42)


param_grid = {
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 10, 20],
    "ccp_alpha": [0.0, 0.01],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Decision Tree"] = grid_search.best_estimator_

Best parameters:  {'ccp_alpha': 0.0, 'max_depth': 5, 'min_samples_split': 2}
Best CV score: 0.615324


###2. Bagging

In [34]:
model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                         n_jobs=-1,
                         random_state=42)


param_grid = {
    "n_estimators": [25, 50]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Bagging"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50}
Best CV score: 0.629151


###3. Random Forest

In [35]:
model = RandomForestClassifier(n_jobs=-1, random_state=42)


param_grid = {
    "n_estimators": [25, 50],

    # 기본값인 "sqrt" 보다 더 적은 max_features를 사용할 때 모델 성능 비교
    "max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Random Forest"] = grid_search.best_estimator_

Best parameters:  {'max_features': 'sqrt', 'n_estimators': 50}
Best CV score: 0.632991


###4. AdaBoost

In [36]:
model = AdaBoostClassifier(random_state=42)


param_grid = {
    #과적합 방지를 위한 비교적 낮은 max_depth의 트리와 성능을 위한 높은 max_depth의 결과를 비교
    "estimator": [DecisionTreeClassifier(max_depth=3), DecisionTreeClassifier(max_depth=6)],
    "learning_rate": [0.1, 1.0],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["AdaBoost"] = grid_search.best_estimator_

Best parameters:  {'estimator': DecisionTreeClassifier(max_depth=6), 'learning_rate': 0.1}
Best CV score: 0.645256


###5. Gradient Boosting

In [37]:
model = GradientBoostingClassifier(random_state=42)

param_grid = {
    "max_depth": [3, 6],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Gradient Boosting"] = grid_search.best_estimator_

Best parameters:  {'max_depth': 3}
Best CV score: 0.649296


###6. XG Boost

In [38]:
model = XGBClassifier(learning_rate=0.1,
                     n_jobs=-1,
                     random_state=42)

param_grid = {
    "n_estimators": [25, 50],

    # 데이터의 feature가 많기 때문에 L1과 L2 정규화를 이용해 더 좋은 성능이 나오는 지 탐색
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["XGBoost"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50, 'reg_alpha': 0.1, 'reg_lambda': 0.1}
Best CV score: 0.652974


###7. Light GBM

In [39]:
model = LGBMClassifier(data_sample_strategy="goss",
                      top_rate=0.2,
                      other_rate=0.1,
                      force_col_wise=True,
                      verbosity=0,
                      n_jobs=-1,
                      random_state=42)

#Light GBM 모델이 특수기호가 있으면 읽지 못해 없애는 과정이 필요함
X_train_lgbm = X_train.copy()
X_val_lgbm = X_val.copy()
X_train_lgbm.columns = X_train_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)
X_val_lgbm.columns = X_val_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)

param_grid = {
    "n_estimators": [25, 50],

    # 데이터의 feature가 많기 때문에 L1과 L2 정규화를 이용해 더 좋은 성능이 나오는 지 탐색
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],

    # feature를 bundle로 처리할 때 달라지는 성능과 속도를 비교
    "enable_bundle": [True, False]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train_lgbm, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["LightGBM"] = grid_search.best_estimator_

Best parameters:  {'enable_bundle': True, 'n_estimators': 25, 'reg_alpha': 0, 'reg_lambda': 0.1}
Best CV score: 0.645042


In [43]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

for _name, _model in models.items():
    y_pred = _model.predict(X_val)
    y_proba = _model.predict_proba(X_val)[:, 1]

    f1 = f1_score (y_val["y"], y_pred)
    accuracy = accuracy_score(y_val["y"], y_pred)
    AUC = roc_auc_score (y_val["y"], y_proba)
    Main_Evaluation = (f1 + accuracy + AUC)/3
    print("{:>17}: f1 = {:.4f} | accuracy = {:.4f} | AUC = {:.4f} | Main Evaluation = {:.4f}".format(
        _name, f1, accuracy, AUC, Main_Evaluation))



    Decision Tree: f1 = 0.6275 | accuracy = 0.6158 | AUC = 0.6543 | Main Evaluation = 0.6325
          Bagging: f1 = 0.6147 | accuracy = 0.6302 | AUC = 0.6868 | Main Evaluation = 0.6439
    Random Forest: f1 = 0.6258 | accuracy = 0.6409 | AUC = 0.6990 | Main Evaluation = 0.6552
         AdaBoost: f1 = 0.6438 | accuracy = 0.6544 | AUC = 0.7121 | Main Evaluation = 0.6701
Gradient Boosting: f1 = 0.6471 | accuracy = 0.6539 | AUC = 0.7141 | Main Evaluation = 0.6717
          XGBoost: f1 = 0.6477 | accuracy = 0.6518 | AUC = 0.7134 | Main Evaluation = 0.6710
         LightGBM: f1 = 0.6384 | accuracy = 0.6492 | AUC = 0.7012 | Main Evaluation = 0.6629


In [41]:
X_test = pd.read_csv('../datasets/X_test_re.csv')
id = pd.read_csv('../datasets/X_test.csv')['id']
X_test

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,average_token_length,num_keywords,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_sentiment_polarity,data_channel,weekday
0,-2.100808,700.181754,0.322256,0.009997,0.530628,NaN,-2.407656,-0.009957,-0.012838,-0.112933,...,0.222965,-1.362449,0.041323,-0.043679,0.014281,-0.886499,-0.358652,-0.754022,World,Tuesday
1,0.776848,606.811584,-0.484049,0.009997,-0.717983,NaN,-2.407656,NaN,0.057192,-1.672762,...,-0.998320,-0.378930,-0.512390,-1.744159,-0.436882,NaN,-0.358652,-0.754022,Business,Tuesday
2,-1.621198,536.783957,0.026030,NaN,0.046546,27.059631,10.127863,NaN,NaN,NaN,...,NaN,-0.870689,NaN,NaN,0.014281,-0.886499,-0.358652,-0.754022,Tech,Thursday
3,NaN,255.214540,1.702155,0.009997,0.877553,40.917366,NaN,-0.009957,0.449549,-0.112933,...,0.222965,-1.854208,0.674138,0.901033,0.529895,-0.886499,-0.358652,-0.754022,Entertainment,Monday
4,NaN,427.365790,0.546700,0.009997,0.744932,3.963405,0.726224,-0.009957,-0.216934,-1.152819,...,0.222965,-1.362449,0.516949,0.853797,0.014281,NaN,0.100619,-0.395070,Entertainment,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9510,-0.661980,2348.748807,-1.139547,0.009997,-0.547322,262.641137,13.261742,12.978783,-0.254695,1.446897,...,-0.692999,NaN,0.337899,0.145264,-0.436882,-0.886499,-0.358652,-0.754022,World,Wednesday
9511,NaN,1523.006372,-0.692725,0.009997,-0.439059,50.155857,-2.407656,-0.009957,0.334047,1.446897,...,0.222965,1.096349,0.524977,0.145264,0.014281,NaN,1.886670,1.000853,Lifestyle,Friday
9512,0.776848,469.674148,-0.017340,0.009997,-0.301827,NaN,10.127863,1.288917,-0.096528,-0.112933,...,1.111172,-0.378930,-0.242632,0.145264,0.014281,-0.886499,-0.358652,-0.754022,Tech,Monday
9513,0.297239,653.496669,0.187511,0.009997,0.597953,17.821140,6.993983,NaN,0.720913,-0.632876,...,0.222965,1.096349,0.420607,0.145264,0.014281,0.453121,NaN,0.131392,Entertainment,Monday


In [42]:
best_model = models["Gradient Boosting"]
predictions = best_model.predict(X_test)
prob_predictions = best_model.predict_proba(X_test)[:, 1]

pd.DataFrame({
    'id': id,
    'probability': prob_predictions,
    'prediction': predictions
}).to_csv('../datasets/prediction.csv', index=False)

ValueError: could not convert string to float: 'World'